## 변수 간 관계 분석

In [92]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import math
import platform

df = pd.read_csv(r'C:\Users\컴퓨터\Documents\[부캠][basic] 심화프로젝트\2025_Airbnb_NYC_listings.csv') #----- 자기 경로 설정!!
df_cleaned = pd.read_csv(r'C:\Users\컴퓨터\Documents\data-analysis-project-BBB\data\first_clean_data.csv')

In [93]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝
from sklearn.preprocessing import MinMaxScaler

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


# 머신러닝 전용 df 전처리

In [94]:
df_machine = df_cleaned.copy()
df_machine.head()

,Unnamed: 0,id,name,description,host_id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bedrooms,beds,amenities,price,availability_365,number_of_reviews,number_of_reviews_ltm,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,log_price
0,0,36121,Lg Rm in Historic Prospect Heights,Cozy space share in the heart of a great neigh...,62165,2009-12-11,-1.0,-1.0,87.0,False,Prospect Heights,Brooklyn,40.673760,-73.966110,Private room in rental unit,Private room,1,1.0,1.0,"[""Refrigerator"", ""Dishes and silverware"", ""Wif...",200,362,9,0,0,0.0,4.88,5.00,4.80,5.00,5.00,5.00,5.00,1,0,1,0,0.05,5.303305
1,1,36647,"1 Bedroom & your own Bathroom, Elevator Apartment",Private bedroom with your own bathroom in a 2 ...,157798,2010-07-04,-1.0,-1.0,100.0,False,East Harlem,Manhattan,40.792454,-73.940742,Private room in condo,Private room,2,1.0,1.0,"[""Oven"", ""Blender"", ""Luggage dropoff allowed"",...",82,204,102,0,0,0.0,4.77,4.82,4.76,4.88,4.90,4.38,4.71,1,0,1,0,0.58,4.418841
2,2,38663,Luxury Brownstone in Boerum Hill,"Beautiful, large home in great hipster neighbo...",165789,2010-07-13,3.0,100.0,40.0,False,Boerum Hill,Brooklyn,40.684420,-73.980680,Private room in home,Private room,2,5.0,5.0,"[""Portable fans"", ""Oven"", ""Baking sheet"", ""Fir...",765,326,43,0,0,0.0,4.70,4.83,4.52,4.88,4.88,4.86,4.62,1,0,1,0,0.28,6.641182
3,3,38833,Spectacular West Harlem Garden Apt,This is a very large and unique space. An inc...,166532,2010-07-14,4.0,100.0,97.0,True,Harlem,Manhattan,40.818058,-73.946671,Entire home,Entire home/apt,2,1.0,1.0,"[""Fire extinguisher"", ""Clothing storage: close...",139,25,241,42,255,35445.0,4.85,4.87,4.50,4.96,4.96,4.79,4.82,1,1,0,0,1.36,4.941642
4,4,39282,“Work-from-home” from OUR home.,*Monthly Discount will automatically apply <br...,168525,2010-07-16,4.0,100.0,100.0,True,Williamsburg,Brooklyn,40.710651,-73.950874,Private room in rental unit,Private room,2,1.0,1.0,"[""Oven"", ""Rice maker"", ""Laundromat nearby"", ""L...",130,38,274,12,154,20020.0,4.82,4.83,4.61,4.94,4.88,4.85,4.78,2,0,2,0,1.54,4.875197


## 컬럼 드랍

In [95]:
drop_cols = ['name', 'description', 'neighbourhood_cleansed', 'number_of_reviews', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication',
            'review_scores_location', 'review_scores_value','estimated_occupancy_l365d','estimated_revenue_l365d','calculated_host_listings_count_shared_rooms','reviews_per_month',
            'calculated_host_listings_count','calculated_host_listings_count_entire_homes','calculated_host_listings_count_private_rooms','host_id','host_since','id','latitude',
            'longitude','amenities','property_type']
df_machine=df_machine.drop(columns = drop_cols)

In [96]:
df_machine.shape

(22248, 16)

## 'host_is_superhost', 'neighbourhood_group_cleansed' 'room_type' 값 drop

### 'host_is_superhost' unknown 값 삭제

In [97]:
host_is_idx = df_machine.loc[df_machine['host_is_superhost'] == 'unknown'].index
df_machine = df_machine.drop(host_is_idx)

In [98]:
df_machine['host_is_superhost'].value_counts()

host_is_superhost
False    15746
True      6129
Name: count, dtype: int64

### 'neighbourhood_group_cleansed' Bronx, Staten Island 값 삭제

In [99]:
neighborhood_idx = df_machine.loc[df_machine['neighbourhood_group_cleansed'].isin(['Bronx','Staten Island'])].index
df_machine = df_machine.drop(neighborhood_idx)

In [100]:
df_machine['neighbourhood_group_cleansed'].value_counts()

neighbourhood_group_cleansed
Manhattan    10008
Brooklyn      7328
Queens        3325
Name: count, dtype: int64

### 'room_type' Hotel room, Shared room

In [101]:
room_idx = df_machine.loc[df_machine['room_type'].isin(['Hotel room','Shared room'])].index
df_machine = df_machine.drop(room_idx)

In [102]:
df_machine['room_type'].value_counts()

room_type
Entire home/apt    11868
Private room        8358
Name: count, dtype: int64

In [103]:
df_machine.shape

(20226, 16)

In [39]:
# df_machine = df_machine.dropna(subset='review_scores_rating')

In [40]:
# df_machine['review_scores_rating'].isna().sum()

In [41]:
# df_machine

------------------------------------

---------------------------------------

# 머신러닝 시작..!

## 1. Train/Test Split: train_test_split 

In [105]:
import pandas as pd
from sklearn.model_selection import train_test_split

X = df_machine.drop(columns=["review_scores_rating"])
y = df_machine["review_scores_rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

----------------------------

# 2. 결측치(Missing): 채우기 + 결측 자체도 피처로

### 2-1. 결측 플래그 만들기

In [106]:
col = "review_scores_accuracy"
X_train[f"{col}_isna"] = X_train[col].isna().astype(int)
X_test[f"{col}_isna"]  = X_test[col].isna().astype(int)

-------------------------------

# 3. 범주형 인코딩(Encoding)

In [107]:
# 2. 원 핫 인코딩
cat_cols = ["neighbourhood_group_cleansed", "room_type","host_is_superhost"]
X_train_dum = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test_dum = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)
# 3. reindex(=컬럼 정렬)
X_test_dum = X_test_dum.reindex(columns=X_train_dum.columns, fill_value=0)

In [108]:
X_train_dum.columns

Index(['Unnamed: 0', 'host_response_time', 'host_response_rate',
       'host_acceptance_rate', 'accommodates', 'bedrooms', 'beds', 'price',
       'availability_365', 'number_of_reviews_ltm', 'review_scores_accuracy',
       'log_price', 'review_scores_accuracy_isna',
       'neighbourhood_group_cleansed_Manhattan',
       'neighbourhood_group_cleansed_Queens', 'room_type_Private room',
       'host_is_superhost_True'],
      dtype='str')

In [109]:
X_test_dum.columns

Index(['Unnamed: 0', 'host_response_time', 'host_response_rate',
       'host_acceptance_rate', 'accommodates', 'bedrooms', 'beds', 'price',
       'availability_365', 'number_of_reviews_ltm', 'review_scores_accuracy',
       'log_price', 'review_scores_accuracy_isna',
       'neighbourhood_group_cleansed_Manhattan',
       'neighbourhood_group_cleansed_Queens', 'room_type_Private room',
       'host_is_superhost_True'],
      dtype='str')

두 컬럼 일치

# 4. 스케일링

- (가격만 로그변환)

In [110]:
X_train_dum.info()

<class 'pandas.DataFrame'>
Index: 16180 entries, 1105 to 17504
Data columns (total 17 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Unnamed: 0                              16180 non-null  int64  
 1   host_response_time                      16180 non-null  float64
 2   host_response_rate                      16180 non-null  float64
 3   host_acceptance_rate                    16180 non-null  float64
 4   accommodates                            16180 non-null  int64  
 5   bedrooms                                16180 non-null  float64
 6   beds                                    16180 non-null  float64
 7   price                                   16180 non-null  int64  
 8   availability_365                        16180 non-null  int64  
 9   number_of_reviews_ltm                   16180 non-null  int64  
 10  review_scores_accuracy                  11275 non-null  float64
 11  lo

In [90]:
# price는 극단값이 꽤 있어서 로그 변환을 같이 쓰는 경우가 많음
X_train_dum["price_log1p"] = np.log1p(X_train_dum["price"])  # log(1+price)
X_test_dum["price_log1p"] = np.log1p(X_test_dum["price"])

In [111]:
from sklearn.preprocessing import StandardScaler

scale_cols = ["price_log1p", "host_response_time", "host_response_rate", "host_acceptance_rate","accommodates","bedrooms","beds","availability_365","number_of_reviews_ltm","review_scores_accuracy",
            "review_scores_accuracy_isna"]
scale_cols = [c for c in scale_cols if c in X_train_dum.columns]  # 안전장치

scaler = StandardScaler()
X_train_dum[scale_cols] = scaler.fit_transform(X_train_dum[scale_cols])
X_test_dum[scale_cols] = scaler.transform(X_test_dum[scale_cols])

X_train_dum[scale_cols].describe()

,host_response_time,host_response_rate,host_acceptance_rate,accommodates,bedrooms,beds,availability_365,number_of_reviews_ltm,review_scores_accuracy,review_scores_accuracy_isna
count,1.618000e+04,1.618000e+04,1.618000e+04,1.618000e+04,1.618000e+04,1.618000e+04,1.618000e+04,1.618000e+04,1.127500e+04,1.618000e+04
mean,4.259743e-17,-7.333785e-17,8.519486e-17,1.053957e-17,5.269785e-17,-2.151829e-17,-3.688850e-17,1.317446e-17,-8.646250e-16,-4.391488e-17
std,1.000031e+00,1.000031e+00,1.000031e+00,1.000031e+00,1.000031e+00,1.000031e+00,1.000031e+00,1.000031e+00,1.000044e+00,1.000031e+00
min,-1.850183e+00,-1.836138e+00,-3.036059e+00,-9.424657e-01,-1.419897e+00,-1.378576e+00,-2.117460e+00,-2.319058e-01,-8.593503e+00,-6.595705e-01
25%,-2.912078e-01,-1.839302e-01,-2.744451e-01,-4.444060e-01,-3.351465e-01,-5.323618e-01,-7.911284e-01,-2.319058e-01,-1.246450e-01,-6.595705e-01
50%,7.481088e-01,6.178766e-01,3.008911e-01,-4.444060e-01,-3.351465e-01,-5.323618e-01,2.681562e-01,-2.319058e-01,2.873535e-01,-6.595705e-01
75%,7.481088e-01,6.178766e-01,7.611601e-01,5.517135e-01,7.496044e-01,3.138524e-01,1.006985e+00,-1.209268e-01,5.620192e-01,1.516138e+00
max,7.481088e-01,6.178766e-01,7.995159e-01,6.528430e+00,1.485137e+01,3.416242e+01,1.131607e+00,6.576358e+01,5.620192e-01,1.516138e+00
